## **Importing libraries**

In [ ]:
from bs4 import BeautifulSoup
import csv
import requests
import time
import hashlib
import random

## **Checking if the ad is unique**

In [ ]:
def extract_info_from_ad(ad):

    info = {}
    details = ad.find('div', class_='pb-2 px-3 z-10 flex-none -mt-1')

    try:
        title = details.find('h2', class_='card-title font-arabic text-sm font-medium leading-5 text-gray-800 max-w-min min-w-full line-clamp-2 my-2')
        info['title'] = title.get_text(strip = True) if title else ''

        price = details.find('data', class_='font-bold font-arabic text-red-600  undefined')
        info['price'] = price.get_text(strip = True) if price else ''

        location = details.find('span', class_='line-clamp-1 truncate text-3xs md:text-xs lg:text-xs w-3/5 font-medium text-neutral-500')
        info['location'] = location.get_text(strip = True).split(',')[0] if location else ''

        return info
    except Exception as e:
        print(f"Error extracting ad info: {e}")
        return None


In [ ]:
def load_seen_hashes(filename='seen_ads_tayara.txt'):
    try:
        with open(filename, 'r') as f:
            return set(f.read().splitlines())
    except FileNotFoundError:
        return set()

def save_seen_hashes(hashes, filename='seen_ads_tayara.txt'):
    with open(filename, 'w') as f:
        for h in hashes:
            f.write(h + '\n')

In [ ]:
seen_ad_hashes = load_seen_hashes()

def hash_ad_content(ad_info):
    key = f"{ad_info['title']}-{ad_info['price']}-{ad_info['location']}"
    return hashlib.md5(key.encode()).hexdigest()

In [ ]:
def is_duplicate_ad(ad_data):

    ad_hash = hash_ad_content(ad_data)

    if ad_hash in seen_ad_hashes:
        return True
    seen_ad_hashes.add(ad_hash)
    return False

## **Sraping Links from Each Website**

In [ ]:
def scrape_unique_ads(ads, page):
    try: #handling for missing elements

        links = []
        duplicates_found = 0
        base_url = "https://www.tayara.tn/"

        for ad in ads:
            a_tag = ad.find('a')
            ad_data = extract_info_from_ad(ad)
            if a_tag and ad_data:
                if not is_duplicate_ad(ad_data):
                    link = base_url + a_tag['href']
                    links.append(link)
                else:
                    duplicates_found += 1
                    print(f"Duplicate AD: {ad_data.get('title')[:30]}...")
            else:
                print('ad found without valid link')
        
        print(f"  → Page {page}: {len(links)} unique ads, {duplicates_found} duplicates filtered")
        return links
    except:
        return None
            

## **Scraping all pages from the website**

In [ ]:
def scrape_all_pages():
    links = []
    page = 1
    seen_ads = set()
    duplicate_pages = 0
    last_page = 300
    
    while page < last_page:
        print(f"Scraping page {page}...")
        url = f'https://www.tayara.tn/ads/c/V%C3%A9hicules/Voitures/?minPrice=4&maxPrice=900000&page={page}'
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            response = requests.get(url, timeout = 10, headers = headers)
            if response.status_code == 500:
                print(f"Reached 500 error on page {page}. STOPPING.")
                break
            response.raise_for_status() #Raise error for status code like 404...
        except requests.exceptions.RequestException as e:
            print(f"Request failed on page {page}: {e}")
            time.sleep(3)
            continue

        if 'Pas de résultats pour votre recherche' in response.text:
            print("No Ads Found - STOPPING.")
            break

        soup = BeautifulSoup(response.text, 'lxml')
        page_scraped = soup.find('div', class_='relative -z-40')
        if page_scraped:
            ads = page_scraped.find_all('article', class_='mx-0') # was ad, instead of page_scraped
        else:
            print('Could not find ads container')
            break

        if len(ads) > 50:
            print(f"Too many ads on page {page}: ({len(ads)}) - STOPPING.")
            break

        # Get ad titles to check for duplicates
        current_page_ads = set()
        for ad in ads:
            title_elem = ad.find('h2')
            if title_elem:
                title = title_elem.get_text().strip()
                current_page_ads.add(title)

        # Check if we've seen these ads before
        if current_page_ads.issubset(seen_ads) and current_page_ads:
            duplicate_pages += 1
            print(f"Duplicate page detected ({duplicate_pages}/2)")
            
            if duplicate_pages >= 2:
                print("Too many duplicate pages - STOPPING.")
                break
        else:
            duplicate_pages = 0  # Reset counter
            seen_ads.update(current_page_ads)

        page_links = scrape_unique_ads(ads, page)
        links.extend(page_links)
        
        print(f"Found {len(page_links)} links on page {page}")

        page += 1
        time.sleep(random.uniform(1,3))

    return links

## **File Creation**

In [ ]:
def save_links_to_csv(links, filename="links_tayara.csv"):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["link"])  # header
        for link in links:
            writer.writerow([link])

In [ ]:
links = scrape_all_pages()

In [ ]:
save_links_to_csv(links)